# Loading

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import os

In [ ]:
adata = sc.read_h5ad("fetal_brain_data_ChP/mhb.h5ad")
adata

In [ ]:
import json
with open('Output/MHB/MHB/tmp_files/regulons.json', 'r', encoding='utf-8') as file:
    grn_dict = json.load(file)
grn_dict.keys()

# Celltype speicifc TF expression

In [ ]:
TF_list = []
for TF in grn_dict.keys():
    TF_list.append(TF.split('(')[0])
adata_gene = adata[:,TF_list]
adata_gene

In [ ]:
adata_gene.layers["counts"] = adata_gene.X.copy()
adata_gene.obs_names_make_unique()
sc.pp.normalize_total(adata_gene)
sc.pp.log1p(adata_gene)
sc.pp.scale(adata_gene)

In [ ]:
sc.tl.dendrogram(adata_gene,'dmt_leiden_anno',use_rep='latent_embeddings_all_spatial_pretrain')
sc.tl.rank_genes_groups(adata_gene, 'dmt_leiden_anno', use_rep='latent_embeddings_all_spatial_pretrain',
                        method='wilcoxon',use_raw=False,key_added='leiden_wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_gene,groupby='dmt_leiden_anno',
                                cmap='Spectral_r',key='leiden_wilcoxon',
                                standard_scale='var',n_genes=3,dendrogram=True)

In [ ]:
n_top = 5
top3_genes_dict = {name: adata_gene.uns['leiden_wilcoxon']['names'][name][:n_top] for 
                   name in adata_gene.uns['leiden_wilcoxon']['names'].dtype.names}
top3_genes_dict

In [ ]:
s = pd.Series(top3_genes_dict)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['celltype', 'tf']
df.to_csv("Output/MHB/celltype_specific_tf.csv")
df

In [ ]:
sc.pl.matrixplot(
    adata_gene,
    top3_genes_dict,
    "dmt_leiden_anno",
    dendrogram=True,
    colorbar_title="mean z-score",
    standard_scale = 'var',
    vmin=-1,
    vmax=1,
    cmap="RdBu_r",
)

# AUC visualization

In [ ]:
auc_mtx = sc.read_h5ad("Process_Data/specific_region/auc_mhb.h5ad")

In [ ]:
sc.tl.dendrogram(auc_mtx,'dmt_leiden_anno',use_rep='latent_embeddings_all_spatial_pretrain')
sc.tl.rank_genes_groups(auc_mtx, 'dmt_leiden_anno', use_rep='latent_embeddings_all_spatial_pretrain',
                        method='wilcoxon',use_raw=False,key_added='leiden_wilcoxon')
sc.pl.rank_genes_groups_dotplot(auc_mtx,groupby='dmt_leiden_anno',
                                cmap='Spectral_r',key='leiden_wilcoxon',jjj
                                standard_scale='var',n_genes=3,dendrogram=True)

In [ ]:
sc.pl.rank_genes_groups_dotplot(auc_mtx,groupby='dmt_leiden_anno',
                                cmap='Spectral_r',key='leiden_wilcoxon',
                                standard_scale='var',n_genes=3,dendrogram=True)

In [ ]:
 nnnmbbbjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjjn_top = 10
top3_genes_dict = {name: adata_gene.uns['leiden_wilcoxon']['names'][name][:n_top]+'(+)' for 
                   name in adata_gene.uns['leiden_wilcoxon']['names'].dtype.names}
s = pd.Series(top3_genes_dict)
s_exploded = s.explode()
df = s_exploded.reset_index()
df.columns = ['celltype', 'grn']
df.to_csv("Output/MHB/celltype_specific_grn.csv")
df